# Notebook 03 — Model Training

Trains two sequence models to predict satellite positions at T+10, T+30, T+60, and T+120 minutes from a 90-minute observation window:

1. **LSTM** — bidirectional LSTM encoder followed by a fully-connected head
2. **Transformer** — sinusoidal positional encoding + encoder-only Transformer, FC head

**Input:** `.npy` arrays + `scaler_X.pkl` from Notebook 02  
**Output:** `lstm_orbit.pt`, `transformer_orbit.pt` saved to `data/collected/` (or `/content/` on Colab)

---

### Colab GPU tip
Enable GPU: **Runtime → Change runtime type → T4 GPU**.  
Both models train in under 20 minutes on T4 for a 5-satellite, 180-day dataset.

## 0 — Environment Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Running on Google Colab')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'plotly'], check=True)
else:
    print('Running locally')

import os, io, zipfile, json, math, pickle, time
from pathlib import Path
from copy import deepcopy

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/content') if IN_COLAB else Path('../data/collected')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 1 — Load Data

**Colab:** upload `dataset.zip` from Notebook 02.  
**Local:** arrays are loaded from `data/collected/` automatically.

In [ ]:
if IN_COLAB:
    _needed = ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy',
               'X_test.npy', 'y_test.npy', 'scaler_X.pkl', 'dataset_meta.json']
    _missing = [f for f in _needed if not (DATA_DIR / f).exists()]
    if _missing:
        from google.colab import files as _colab_files
        print('Upload the dataset.zip from Notebook 02.')
        _uploaded = _colab_files.upload()
        _zip_bytes = next(iter(_uploaded.values()))
        with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as zf:
            zf.extractall(DATA_DIR)
        print('Extracted.')

# Load arrays
X_train = np.load(DATA_DIR / 'X_train.npy')
y_train = np.load(DATA_DIR / 'y_train.npy')
X_val   = np.load(DATA_DIR / 'X_val.npy')
y_val   = np.load(DATA_DIR / 'y_val.npy')
X_test  = np.load(DATA_DIR / 'X_test.npy')
y_test  = np.load(DATA_DIR / 'y_test.npy')

with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

N_FEATURES  = X_train.shape[-1]
INPUT_LEN   = X_train.shape[1]
N_HORIZONS  = y_train.shape[1]   # 4
N_TARGETS   = y_train.shape[2]   # 3 (lat, lon, alt)
N_OUT       = N_HORIZONS * N_TARGETS  # 12

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'Features: {N_FEATURES}  Horizons: {N_HORIZONS}  Targets/horizon: {N_TARGETS}')

## 2 — Hyperparameters & DataLoaders

In [ ]:
HIDDEN        = 256
LSTM_LAYERS   = 2
TF_HEADS      = 4
TF_ENC_LAYERS = 2
TF_FF_DIM     = 512
DROPOUT       = 0.1
EPOCHS        = 100
BATCH         = 128
LR            = 1e-3
PATIENCE      = 10  # early stopping patience

def make_loader(X, y, batch, shuffle):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y.reshape(len(y), -1), dtype=torch.float32)  # flatten to (N, 12)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=shuffle, pin_memory=True)

train_dl = make_loader(X_train, y_train, BATCH, shuffle=True)
val_dl   = make_loader(X_val,   y_val,   BATCH, shuffle=False)
test_dl  = make_loader(X_test,  y_test,  BATCH, shuffle=False)
print(f'Train batches: {len(train_dl)}  Val batches: {len(val_dl)}  Test batches: {len(test_dl)}')

## 3 — Model Definitions

Three models in increasing representational capacity:
1. **MLP** — flattens the 90-step window, no temporal structure (ablation baseline)
2. **LSTM** — bidirectional recurrent encoder (learns sequential dynamics)
3. **Transformer** — self-attention encoder (learns long-range patterns)

The MLP establishes whether temporal structure is actually needed. If LSTM/Transformer only marginally beat MLP, the task may not require sequence modeling.

### 3a — MLP (Flat Baseline)

In [ ]:
class MLPOrbit(nn.Module):
    """
    Flat MLP — no temporal structure.
    Input: (B, T, F) flattened to (B, T*F).
    Ablation baseline to test whether sequence modeling adds value.
    """
    def __init__(self, input_len, n_features, hidden, n_out, dropout):
        super().__init__()
        in_dim = input_len * n_features
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden * 2),
            nn.LayerNorm(hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x):
        # x: (B, T, F) → flatten → (B, T*F)
        return self.net(x.flatten(start_dim=1))

mlp_model = MLPOrbit(INPUT_LEN, N_FEATURES, HIDDEN, N_OUT, DROPOUT).to(DEVICE)
n_params_mlp = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f'MLP parameters: {n_params_mlp:,}')

In [ ]:
### 3b — LSTM Model

Bidirectional LSTM encoder → flatten last hidden state → two-layer FC head → 12 outputs.

### 3b — Transformer Model

Sinusoidal positional encoding → `nn.TransformerEncoder` → mean-pool sequence → two-layer FC head → 12 outputs.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerOrbit(nn.Module):
    def __init__(self, n_features, d_model, nhead, num_encoder_layers, ff_dim, n_out, dropout):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, ff_dim,
                                               dropout=dropout, batch_first=True,
                                               norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_out),
        )

    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))
        x = self.encoder(x)
        ctx = x.mean(dim=1)   # mean-pool over sequence
        return self.head(ctx)

tf_model = TransformerOrbit(N_FEATURES, HIDDEN, TF_HEADS, TF_ENC_LAYERS, TF_FF_DIM, N_OUT, DROPOUT).to(DEVICE)
n_params_tf = sum(p.numel() for p in tf_model.parameters() if p.requires_grad)
print(f'Transformer parameters: {n_params_tf:,}')

## 4 — Training Loop

Shared training loop used for both models.  
Optimiser: **AdamW** | LR schedule: **cosine annealing** | Early stopping on validation loss.

In [ ]:
def train_model(model, train_dl, val_dl, epochs, lr, patience, name):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    best_val, best_state, wait = float('inf'), None, 0
    history = {'train': [], 'val': []}

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        running = 0.0
        for Xb, yb in train_dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * len(Xb)
        scheduler.step()
        train_loss = running / len(train_dl.dataset)

        # --- Validate ---
        model.eval()
        running = 0.0
        with torch.no_grad():
            for Xb, yb in val_dl:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                running += criterion(model(Xb), yb).item() * len(Xb)
        val_loss = running / len(val_dl.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f'[{name}] Epoch {epoch:3d}/{epochs}  train={train_loss:.4f}  val={val_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

        # Early stopping
        if val_loss < best_val:
            best_val = val_loss
            best_state = deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    print(f'[{name}] Best val loss: {best_val:.4f}')
    return history

In [ ]:
print('=== Training MLP (ablation baseline) ===')
mlp_history = train_model(mlp_model, train_dl, val_dl, EPOCHS, LR, PATIENCE, 'MLP')

print('\n=== Training LSTM ===')
lstm_history = train_model(lstm_model, train_dl, val_dl, EPOCHS, LR, PATIENCE, 'LSTM')

print('\n=== Training Transformer ===')
tf_history = train_model(tf_model, train_dl, val_dl, EPOCHS, LR, PATIENCE, 'Transformer')

## 5 — Loss Curves

In [ ]:
plot_history({'MLP': mlp_history, 'LSTM': lstm_history, 'Transformer': tf_history},
             'Train vs Validation Loss — All Models')

## 6 — Save Weights & Download (Colab)

In [ ]:
mlp_path  = DATA_DIR / 'mlp_orbit.pt'
lstm_path = DATA_DIR / 'lstm_orbit.pt'
tf_path   = DATA_DIR / 'transformer_orbit.pt'

torch.save({'model_state': mlp_model.state_dict(),
            'hparams': {'input_len': INPUT_LEN, 'n_features': N_FEATURES,
                        'hidden': HIDDEN, 'n_out': N_OUT, 'dropout': DROPOUT}},
           mlp_path)

torch.save({'model_state': lstm_model.state_dict(),
            'hparams': {'hidden': HIDDEN, 'lstm_layers': LSTM_LAYERS,
                        'n_features': N_FEATURES, 'n_out': N_OUT, 'dropout': DROPOUT}},
           lstm_path)

torch.save({'model_state': tf_model.state_dict(),
            'hparams': {'d_model': HIDDEN, 'nhead': TF_HEADS,
                        'num_encoder_layers': TF_ENC_LAYERS, 'ff_dim': TF_FF_DIM,
                        'n_features': N_FEATURES, 'n_out': N_OUT, 'dropout': DROPOUT}},
           tf_path)

for p in [mlp_path, lstm_path, tf_path]:
    print(f'Saved: {p.name}  ({p.stat().st_size/1024:.1f} KB)')

if IN_COLAB:
    from google.colab import files as _colab_files
    for p in [mlp_path, lstm_path, tf_path]:
        _colab_files.download(str(p))
    print('Downloads started — you will need all three .pt files in Notebook 04.')